# B4-clock — Clock-time-preserving shift null for the external factor

**Purpose.** The main table's null uses *arbitrary* circular shifts of the external factor. A referee can
object that such shifts also break common **intraday phase** (volume/volatility seasonality), so part of the
observed separation might reflect broken clock structure rather than genuine temporal alignment. This
notebook runs the recommended robustness: a null that **permutes whole days** of the factor, so every value
keeps its exact time of day — the intraday clock is preserved by construction — while the day-to-day
alignment with the panel is destroyed.

**Honest resolution limits, stated up front.**
- With $D$ full days in a window there exist at most $D!-1$ distinct clock-preserving replicates. A 4-day
  crisis window gives $23$ replicates, so the **minimum attainable per-window $p$ is $1/24\approx0.042$**.
  The notebook enumerates *all* permutations when $D!-1\le$ the cap, and reports the attainable minimum
  alongside each $p$ so nothing is over-claimed.
- The headline number is therefore the **Fisher combination across the eight disjoint windows** (periods do
  not overlap), computed per threshold. Discrete Monte Carlo $p$-values make the combination conservative.
- Weekday identity is *not* preserved (in a 4-day window every weekday appears once, so preserving it would
  leave no randomization at all); the null preserves time-of-day, which is the dominant crypto seasonality.
- $p$-values use the standard $(b+1)/(B+1)$ correction throughout, consistent with the manuscript.

Requires `crypto_confirmatory_V4.ipynb` next to this notebook. Reuses the exact same factor construction,
fitting functions and robust downloader as the main B4 run; the panel cache is reused (no re-download).

In [ ]:

import json, ast, sys
from pathlib import Path
import numpy as np, pandas as pd

# --- locate the V4 notebook (must sit next to this one) ---
CANDS = sorted(Path.cwd().glob("*V4*CONFIRMATORY*.ipynb")) + \
        sorted(Path.cwd().glob("*crypto*V4*.ipynb")) + \
        sorted(Path.cwd().glob("crypto_confirmatory_V4.ipynb"))
assert CANDS, ("Place crypto_confirmatory_V4.ipynb next to this notebook. "
               "Its functions (load_window_panel, build_events_from_features, "
               "build_drive_matrix, fit_network, build_filtered_histories, "
               "spectral_radius, rank1_share, ...) are reused here.")
V4 = CANDS[0]; print("V4 found:", V4.name)
nb_v4 = json.load(open(V4))
code_cells = [(i, "".join(c["source"])) for i, c in enumerate(nb_v4["cells"])
              if c["cell_type"] == "code"]

def _is_def_cell(src):
    try: tree = ast.parse(src)
    except SyntaxError: return False
    has_def = any(isinstance(n, (ast.FunctionDef, ast.ClassDef)) for n in tree.body)
    top_calls = [n for n in tree.body
                 if isinstance(n, ast.Expr) and isinstance(n.value, ast.Call)]
    return has_def and not top_calls

to_exec = [code_cells[0]] + [(i, s) for i, s in code_cells[1:] if _is_def_cell(s)]
for i, src in to_exec:
    try: exec(compile(src, f"<V4 cell {i}>", "exec"), globals())
    except Exception as exc: print(f"  ! V4 cell {i} skipped ({type(exc).__name__}: {exc})")

# copy-on-write safe PC1 (same numeric result as V4)
_ssd = safe_standardize_df
def first_pc_score(df):
    X = np.array(_ssd(df).to_numpy(dtype=float), copy=True)
    if X.shape[1] == 0: return np.zeros(X.shape[0])
    X = X - X.mean(axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(X, full_matrices=False)
    score = X @ vt[0]
    if np.corrcoef(score, X.mean(axis=1))[0, 1] < 0: score = -score
    return (score - score.mean()) / (score.std() + 1e-12)

for _f in ["load_window_panel","panel_to_wide_features","build_events_from_features",
           "continuous_activity_score","build_drive_matrix","apply_drive_timing",
           "fit_network","build_filtered_histories","spectral_radius","rank1_share",
           "effective_design"]:
    assert _f in globals(), f"missing V4 function: {_f}"
print("V4 pipeline loaded OK.")

# ---- robust downloader: retries + atomic write + zip validation ----
import time as _time, zipfile as _zip, urllib.request, urllib.error

def _valid_zip(p):
    try:
        with _zip.ZipFile(p) as zf:
            return zf.testzip() is None and len(zf.namelist()) > 0
    except Exception:
        return False

def download_one_aggtrade(symbol, date_str, raw_dir=RAW, force=False, timeout=120, retries=5):
    out = raw_dir / symbol / f"{symbol}-aggTrades-{date_str}.zip"
    out.parent.mkdir(parents=True, exist_ok=True)
    if out.exists() and not force:
        if out.stat().st_size > 100 and _valid_zip(out):
            return out
        try: out.unlink()
        except Exception: pass
    url = aggtrade_url(symbol, date_str)
    tmp = out.parent / (out.name + ".part")
    last = None
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as r:
                data = r.read()
            tmp.write_bytes(data)
            if _valid_zip(tmp):
                tmp.replace(out); return out
            try: tmp.unlink()
            except Exception: pass
            last = "corrupt payload"
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None            # symbol-day genuinely absent -> no retry
            last = e
        except Exception as e:         # DNS blips, timeouts, IncompleteRead, ...
            last = e
        _time.sleep(1.5 * (attempt + 1))
    try: tmp.unlink()
    except Exception: pass
    print(f"Download failed after {retries} tries: {symbol} {date_str}: {last}")
    return None
print("robust downloader installed")


## Configuration (same corrected 8-symbol external basket as the successful B4 run)

In [ ]:

# --- panel + cesta externa CORREGIDA (8 pares listados en Binance antes de 2022) ---
PANEL_SYMBOLS = list(SYMBOLS)
EXTERNAL_SYMBOLS = [s for s in [
    "XLMUSDT","EOSUSDT","ALGOUSDT","AAVEUSDT","THETAUSDT","VETUSDT","ENJUSDT","MANAUSDT"
] if s not in PANEL_SYMBOLS]
assert not (set(EXTERNAL_SYMBOLS) & set(PANEL_SYMBOLS)), "external set must be disjoint from panel"

DESIGN = dict(event_type="volume_burst", bin="1min", lags=60, half_life=10)
Q_GRID = [0.95, 0.97]          # los umbrales del headline (Table external)
DRIVE_TIMING = "lag1"
MAX_CLOCK_REPS = 20 if QUICK_MODE else 500   # tope; se enumera TODO si D!-1 <= tope
rng = np.random.default_rng(RANDOM_SEED)
print("panel:", len(PANEL_SYMBOLS), "| external:", EXTERNAL_SYMBOLS)


## Factor construction and fitting (verbatim from B4)

In [ ]:

def external_single_factor(window):
    """Return a pd.Series m(t) indexed by the window's minute grid (PC1 of external activity)."""
    panel_ext = load_window_panel(EXTERNAL_SYMBOLS, window, bin_size=DESIGN["bin"])
    if panel_ext.empty:
        raise RuntimeError(f"no external data for {window['name']}")
    present = [s for s in EXTERNAL_SYMBOLS if s in set(panel_ext["symbol"])]
    feats_ext = panel_to_wide_features(panel_ext, present)
    act = continuous_activity_score(feats_ext)          # T x K_ext, no event indicators
    m = first_pc_score(act)                             # 1-D PC1 score
    return pd.Series(m, index=act.index, name="m_ext")

def broadcast_drive(m_series, index, symbols, timing):
    """Same external factor in every equation -> rank-one loading by construction."""
    m = m_series.reindex(index).ffill().fillna(0.0)
    M_df = pd.DataFrame(np.repeat(m.to_numpy()[:,None], len(symbols), axis=1),
                        index=index, columns=symbols)
    return apply_drive_timing(M_df, timing)

In [ ]:

def fit_pair(Y_df, M_df, lags, half_life):
    Y,H,Mx,idx = effective_design(Y_df, M_df, lags, half_life)
    naive = fit_network(Y, H, M=None)
    drive = fit_network(Y, H, M=Mx)
    dB = naive["B"] - drive["B"]
    U,S,Vt = np.linalg.svd(dB, full_matrices=False)
    u1 = U[:,0]
    return dict(B_naive=naive["B"], B_drive=drive["B"], c=drive["c"], dB=dB, u1=u1,
                rho_naive=spectral_radius(naive["B"]),
                rho_drive=spectral_radius(drive["B"]),
                r1=rank1_share(dB), resid=naive["residuals"], Yeff=Y, Heff=H, Meff=Mx)

def cos(a,b):
    a=np.asarray(a,float); b=np.asarray(b,float)
    return float(abs(a@b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12))

def procedure_null(Y_df, m_series, symbols, lags, half_life, reps, seed):
    """Circular-shift the FACTOR (preserving its autocorrelation), refit, recollect r1 and delta_rho.
    A real common factor should deflate rho far more than a shifted (fake) factor; r1 stays ~1 either way."""
    r = np.random.default_rng(seed)
    idx = Y_df.index
    base = fit_pair(Y_df, broadcast_drive(m_series, idx, symbols, DRIVE_TIMING), lags, half_life)
    T=len(m_series); r1n=[]; drn=[]
    for _ in range(reps):
        sh=int(r.integers(max(30,lags), max(31,T-lags)))
        m_sh = pd.Series(np.roll(m_series.to_numpy(), sh), index=m_series.index)
        f = fit_pair(Y_df, broadcast_drive(m_sh, idx, symbols, DRIVE_TIMING), lags, half_life)
        r1n.append(f["r1"]); drn.append(base["rho_naive"]-f["rho_drive"])
    r1n=np.array(r1n); drn=np.array(drn)
    dr_obs = base["rho_naive"]-base["rho_drive"]
    return dict(r1_obs=base["r1"], r1_null_med=float(np.median(r1n)),
                p_r1=float(np.mean(r1n>=base["r1"])),
                drho_obs=float(dr_obs), drho_null_med=float(np.median(drn)),
                p_drho=float(np.mean(drn>=dr_obs)))

def independent_c(Y_df, m_series, symbols, lags, half_life):
    """Sample split: estimate loading c on the FIRST half, geometry u1(ΔB) on the SECOND half.
    High cos(u1_B, c_A) means the rank-one geometry is recovered, not imposed by one fit."""
    idx=Y_df.index; h=len(idx)//2
    A=Y_df.iloc[:h]; B=Y_df.iloc[h:]
    fA=fit_pair(A, broadcast_drive(m_series, A.index, symbols, DRIVE_TIMING), lags, half_life)
    fB=fit_pair(B, broadcast_drive(m_series, B.index, symbols, DRIVE_TIMING), lags, half_life)
    cA=np.nan_to_num(fA["c"]); 
    return cos(fB["u1"], cA)

## Clock-preserving day permutations and Fisher combination

In [ ]:

import itertools, math

def day_blocks(m_series):
    """Bloques por fecha calendario, en orden; posiciones de los dias COMPLETOS (longitud modal)."""
    idx = m_series.index
    dser = pd.Series([d.date() for d in idx], index=idx)
    order = list(dict.fromkeys(dser))                    # fechas en orden de aparicion
    groups = [(d, m_series[dser == d].to_numpy()) for d in order]
    lengths = [len(v) for _, v in groups]
    full_len = max(set(lengths), key=lengths.count)      # longitud modal = dia completo
    full_pos = [i for i, (_, v) in enumerate(groups) if len(v) == full_len]
    return groups, full_pos

def permute_days(m_series, perm, groups, full_pos):
    """Reensambla la serie permutando SOLO los bloques de dias completos.
    El indice temporal no cambia: cada valor conserva su hora del dia (clock-time preserved)."""
    vals = [v.copy() for _, v in groups]
    src = [groups[full_pos[p]][1] for p in perm]
    for slot, block in zip(full_pos, src):
        vals[slot] = block
    return pd.Series(np.concatenate(vals), index=m_series.index, name=m_series.name)

def clock_perms(D, max_reps, r):
    """Permutaciones no-identidad de range(D): TODAS si D!-1<=max_reps, si no muestreo sin reemplazo."""
    total = math.factorial(D) - 1
    if total <= max_reps:
        return [p for p in itertools.permutations(range(D)) if p != tuple(range(D))], True
    seen = {tuple(range(D))}; out = []
    while len(out) < max_reps:
        p = tuple(int(x) for x in r.permutation(D))
        if p not in seen:
            seen.add(p); out.append(p)
    return out, False

def chi2_sf_even(x, df):
    """Supervivencia chi-cuadrado para df par (Fisher): sin scipy."""
    m = df // 2
    term = math.exp(-x / 2.0); s = term
    for k in range(1, m):
        term *= (x / 2.0) / k; s += term
    return min(1.0, s)


## Run → `tables/b4_clock_shift.csv`

In [ ]:

OUTB = OUT / "tables"; OUTB.mkdir(parents=True, exist_ok=True)
rows = []
for w in WINDOWS:
    try:
        m_ext = external_single_factor(w)
    except Exception as e:
        print("skip", w["name"], e); continue
    panel = load_window_panel(PANEL_SYMBOLS, w, bin_size=DESIGN["bin"])
    if panel.empty: print("skip panel", w["name"]); continue
    feats = panel_to_wide_features(panel, PANEL_SYMBOLS)
    groups, full_pos = day_blocks(m_ext)
    D = len(full_pos)
    perms, exhaustive = clock_perms(D, MAX_CLOCK_REPS,
                                    np.random.default_rng(RANDOM_SEED + hash(w["name"]) % 9999))
    B = len(perms)
    for q in Q_GRID:
        Y_df, _ = build_events_from_features(feats, event_type=DESIGN["event_type"], q=q)
        keep = [s for s in Y_df.columns if Y_df[s].sum() >= MIN_EVENT_COUNT_PER_ASSET]
        if len(keep) < 3: continue
        Y_df = Y_df[keep]
        base = fit_pair(Y_df, broadcast_drive(m_ext, Y_df.index, keep, DRIVE_TIMING),
                        DESIGN["lags"], DESIGN["half_life"])
        drho_obs = base["rho_naive"] - base["rho_drive"]
        nulls = []
        for p in perms:
            m_p = permute_days(m_ext, p, groups, full_pos)
            f = fit_pair(Y_df, broadcast_drive(m_p, Y_df.index, keep, DRIVE_TIMING),
                         DESIGN["lags"], DESIGN["half_life"])
            nulls.append(base["rho_naive"] - f["rho_drive"])
        nulls = np.array(nulls)
        b = int((nulls >= drho_obs).sum())
        p_clock = (b + 1) / (B + 1)                    # correccion Monte Carlo estandar
        rows.append(dict(window=w["name"], kind=w["kind"], q=q, K=len(keep),
                         D_full_days=D, B_reps=B, exhaustive=exhaustive,
                         drho_obs=float(drho_obs), drho_null_med=float(np.median(nulls)),
                         b_exceed=b, p_clock=float(p_clock),
                         p_min_attainable=1.0 / (B + 1)))
        print(f"{w['name']:18s} q={q}  D={D}  B={B}{'(all)' if exhaustive else ''}  "
              f"drho={drho_obs:.3f}  null_med={np.median(nulls):.3f}  "
              f"b={b}  p_clock={p_clock:.3f}")

res = pd.DataFrame(rows); res.to_csv(OUTB / "b4_clock_shift.csv", index=False)

# --- combinacion de Fisher entre ventanas (periodos disjuntos) por umbral ---
print()
for q in Q_GRID:
    ps = res[res.q == q].p_clock.to_numpy()
    W = len(ps)
    if W == 0: continue
    X2 = -2.0 * np.log(ps).sum()
    p_comb = chi2_sf_even(X2, 2 * W)
    print(f"q={q}: Fisher X2={X2:.1f} (df={2*W}) -> combined p = {p_comb:.2e}  "
          f"(per-window p in [{ps.min():.3f},{ps.max():.3f}])")
res.round(3)


## How to report this in the manuscript

The manuscript currently ends the shift-null paragraph with: *"Because arbitrary circular shifts may also
disturb common intraday phase, clock-time-preserving shifts remain a useful additional robustness check."*

After running this notebook (QUICK_MODE=False), replace that sentence with (fill the brackets from the CSV):

> A clock-time-preserving null --- permuting whole days of the factor, which leaves the time-of-day profile
> intact --- gives per-window tail probabilities of at most [MAX p_clock] (the attainable minimum with $D$
> full days is $1/(B{+}1)$, reported alongside each value), and a Fisher combination across the eight
> disjoint windows yields $p=$[p_comb] at each threshold. The temporal-alignment conclusion is therefore
> not an artifact of broken intraday phase.

If some window's $p$ equals its attainable minimum (all permutations below the observed reduction), say
exactly that: "the observed reduction exceeds every clock-preserving replicate", which is the strongest
statement the discrete null permits --- do not report a smaller $p$ than $1/(B+1)$.